In [ ]:
# Cell 1 — Setup
"""
07_temperature.ipynb
====================
Temperature dependence analysis — a core physics-informed advantage claim.

Scripted companions:
- `scripts/experiments/temperature_extrapolation.py` for T-cut generalization studies
- `scripts/evaluation/validate_physics.py` for van't Hoff and parameter validation
- `scripts/evaluation/error_analysis.py` for discussion-oriented failure analysis
- `results/temperature_extrapolation.json` and `results/physics_validation.json`
  for the canonical paper-analysis outputs
"""

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
TABLES_DIR = PROJECT_ROOT / "tables"
NOTEBOOK_FIG_DIR = FIGURES_DIR / "notebooks"
NOTEBOOK_RESULTS_DIR = RESULTS_DIR / "notebooks"
NOTEBOOK_FIG_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from tgnn_solv.inference import load_model, temperature_scan
from tgnn_solv.eval_temperature import (
    temperature_coverage,
    vant_hoff_check,
    evaluate_multi_T_pairs,
)
from tgnn_solv.data import PROCESSED_DIR

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
MODEL_PATH = CHECKPOINT_DIR / "tgnn_solv_trained.pt"
model, cfg = load_model(str(MODEL_PATH), DEVICE)
print(f"Loaded {MODEL_PATH} on {DEVICE}")


## Temperature physics and van't Hoff analysis

In TGNN-Solv, temperature dependence is not a post-hoc fit; it passes through the solver formula

$$
\ln x_2^{\mathrm{phys}}(T) = -\Phi(T) - \ln \gamma_2(T).
$$

In the current implementation, the ideal crystal contribution is

$$
\Phi(T) = \frac{\Delta H_{\mathrm{fus}}}{R}
\left(\frac{1}{T} - \frac{1}{T_m}\right)
- \frac{\Delta C_p}{R}
\left[\left(\frac{T_m}{T} - 1\right) - \ln\!\left(\frac{T_m}{T}\right)\right].
$$

This gives the ideal-solubility term

$$
x_2^{\mathrm{ideal}} = e^{-\Phi(T)}.
$$

If the temperature dependence of activity is moderate, a van't Hoff-like approximation emerges:

$$
\ln x_2 \approx A + \frac{B}{T}.
$$

In `vant_hoff_check(...)`, the linear fit is performed against the axis \(1000/T\), so the effective heat
of solution is reconstructed as

$$
\Delta H_{\mathrm{sol}}^{\mathrm{eff}} = -B \cdot R \cdot 1000.
$$

For physically plausible behavior, the sign of the derivative also matters:

$$
\frac{\partial \ln x_2}{\partial T} \ge 0,
$$

and the decomposition view lets us interpret

$$
\frac{\partial \ln x_2}{\partial T} =
-\frac{\partial \Phi}{\partial T}
-\frac{\partial \ln \gamma_2}{\partial T}
+ \frac{\partial \Delta_{\mathrm{corr}}}{\partial T}
$$

as the sum of crystal, non-ideal, and correction contributions.


## Step 1. Check whether there is enough temperature signal at all

Before analyzing extrapolation or van't Hoff consistency, it is important to understand how many
multi-temperature pairs the dataset actually contains. Otherwise it is easy to demand temperature physics
from a model trained on data concentrated almost entirely around 298 K.


In [ ]:
# Cell 2 — Temperature coverage in dataset
unified = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")
all_data = pd.concat([unified, val_df, test_df], ignore_index=True)

coverage = temperature_coverage(all_data)

## Step 2. Inspect individual curves, not only aggregate metrics

Representative examples are needed to see the shape of the temperature dependence directly.
Even when the aggregate metric is good, individual pairs may show an unnatural curvature,
the wrong slope, or unstable component-wise decomposition.


In [ ]:
# Cell 3 — van't Hoff plots for representative systems

systems = [
    ("CC(=O)Nc1ccc(O)cc1", "CCO", "Paracetamol / Ethanol"),
    ("CC(=O)Nc1ccc(O)cc1", "O", "Paracetamol / Water"),
    ("c1ccc2ccccc2c1", "c1ccccc1", "Naphthalene / Benzene"),
    ("OC(=O)c1ccccc1", "CCO", "Benzoic acid / Ethanol"),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, (sol, slv, title) in zip(axes.flat, systems):
    vH = vant_hoff_check(
        model, sol, slv,
        T_range=(270, 350), n_points=30,
        experimental=all_data,
    )

    scan = vH["scan"]
    inv_T = 1000.0 / scan["T"].values
    ln_x2 = scan["ln_x2"].values

    # Predicted curve
    ax.plot(inv_T, ln_x2, "b-", lw=2, label="TGNN-Solv")

    # Experimental points (if available)
    if "exp_T" in vH:
        exp_inv_T = 1000.0 / vH["exp_T"]
        ax.scatter(exp_inv_T, vH["exp_ln_x2"],
                   c="red", s=50, zorder=5, label="Experimental")

    ax.set_xlabel("1000/T (K⁻¹)")
    ax.set_ylabel("ln(x₂)")
    ax.set_title(f"{title}\n"
                 f"R²(vH)={vH['vH_r2']:.3f}, "
                 f"ΔH_sol={vH['dH_sol_effective']/1000:.1f} kJ/mol")
    ax.legend(fontsize=9)
    ax.invert_xaxis()

plt.tight_layout()
plt.savefig(NOTEBOOK_FIG_DIR / "vant_hoff_plots.png", dpi=150)
plt.show()


## Step 3. Evaluate many multi-temperature pairs

After the hand-picked illustrations, the notebook moves to a larger-scale evaluation over all
available multi-temperature pairs. This is a more honest test of temperature generalization than
selective visualization.


In [ ]:
# Cell 4 — Multi-T pair evaluation

multi_T_results = evaluate_multi_T_pairs(
    model, all_data, min_T_points=3, max_pairs=50,
)

if len(multi_T_results) > 0:
    multi_T_results.to_csv(NOTEBOOK_RESULTS_DIR / "multi_T_pair_results.csv", index=False)
    print("Saved notebook copy to", NOTEBOOK_RESULTS_DIR / "multi_T_pair_results.csv")
    print() 
    print("Top 10 pairs by MAE:")
    print(multi_T_results.sort_values("mae").head(10).to_string(
        index=False, float_format="{:.3f}".format
    ))

    # Slope comparison plot
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(
        multi_T_results["exp_slope"],
        multi_T_results["pred_slope"],
        s=40, alpha=0.7, c="steelblue",
    )
    lims = [
        min(multi_T_results["exp_slope"].min(),
            multi_T_results["pred_slope"].min()) * 1.1,
        max(multi_T_results["exp_slope"].max(),
            multi_T_results["pred_slope"].max()) * 1.1,
    ]
    ax.plot(lims, lims, "r--", lw=1)
    ax.set_xlabel("Experimental d(ln x₂)/d(1/T)")
    ax.set_ylabel("Predicted d(ln x₂)/d(1/T)")
    ax.set_title("Temperature dependence: slope parity")
    ax.set_aspect("equal")
    plt.tight_layout()
    plt.savefig(NOTEBOOK_FIG_DIR / "T_slope_parity.png", dpi=150)
    plt.show()


## Step 4. Identify which component is moving the curve

In the final step, it is useful to separate the ideal crystal term, the non-ideal activity-coefficient
contribution, and the correction branch. Otherwise one can observe the “right” increase of solubility with
temperature without understanding what actually caused it.


In [ ]:
# Cell 5 — Decomposition: why does solubility increase with T?

from tgnn_solv.inference import predict_solubility

sol_smi = "CC(=O)Nc1ccc(O)cc1"  # paracetamol
slv_smi = "CCO"                  # ethanol

# Collect all data in one pass
T_values = np.linspace(270, 350, 30)
records = []
for T_val in T_values:
    r = predict_solubility(model, sol_smi, slv_smi, float(T_val))
    records.append(r)

scan_df = pd.DataFrame(records)
T_celsius = scan_df["T"].values - 273.15

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Total solubility
ax = axes[0]
ax.plot(T_celsius, scan_df["ln_x2"], "o-", color="steelblue", ms=4)
ax.set_xlabel("T (°C)")
ax.set_ylabel("ln(x₂)")
ax.set_title("Total solubility")

# Decomposition: ideal + non-ideal + correction
ax = axes[1]
Phi_vals = -scan_df["Phi"].values
lng_vals = -scan_df["ln_gamma_2"].values
corr_vals = scan_df["correction"].values

ax.plot(T_celsius, Phi_vals, "s-", label="−Φ (crystal)", color="coral", ms=4)
ax.plot(T_celsius, lng_vals, "^-", label="−ln(γ₂)", color="seagreen", ms=4)
ax.plot(T_celsius, corr_vals, "d-", label="correction", color="gray", ms=4)
ax.set_xlabel("T (°C)")
ax.set_ylabel("Contribution to ln(x₂)")
ax.set_title("Decomposition vs temperature")
ax.legend(fontsize=9)

# γ₂ vs T
ax = axes[2]
ax.plot(T_celsius, scan_df["gamma_2"], "o-", color="purple", ms=4)
ax.set_xlabel("T (°C)")
ax.set_ylabel("γ₂")
ax.set_title("Activity coefficient vs T")

plt.tight_layout()
plt.savefig(NOTEBOOK_FIG_DIR / "T_decomposition.png", dpi=150)
plt.show()
